# DPO Mini Project: Preference Tuning with Chosen/Rejected Pairs

**Goal:** run a tiny Direct Preference Optimization experiment on hand-written preference pairs.

This notebook is intentionally small: it demonstrates the RLHF-style preference loop without building a reward model or PPO pipeline.


In [ ]:
!pip -q install -U "transformers>=4.45" "datasets>=2.20" "accelerate>=0.33" "peft>=0.12" "trl>=0.11" pandas


In [ ]:
import inspect
import random

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType
from trl import DPOConfig, DPOTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


In [ ]:
# Small enough for Colab. If you have more VRAM, try "Qwen/Qwen2.5-0.5B-Instruct".
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

preference_rows = [
    {
        "prompt": "Explain LoRA to a software engineer in two sentences.",
        "chosen": "LoRA fine-tunes a model by adding small trainable low-rank matrices to selected layers while keeping the base weights frozen. This greatly reduces trainable parameters and makes experiments cheaper.",
        "rejected": "LoRA is just when you train the whole model with a smaller learning rate.",
    },
    {
        "prompt": "Why evaluate retrieval before generation in RAG?",
        "chosen": "Because generation quality depends on whether the retriever surfaced the right evidence. If the answer is not in the retrieved context, the model is likely guessing even if the prose sounds confident.",
        "rejected": "Retrieval evaluation is unnecessary because the language model can usually infer missing facts.",
    },
    {
        "prompt": "Compare DPO and PPO-style RLHF briefly.",
        "chosen": "DPO optimizes directly from chosen and rejected responses, avoiding a separate reward model and online RL loop. PPO-style RLHF usually trains a reward model, then updates the policy with reinforcement learning.",
        "rejected": "DPO and PPO are the same algorithm with different names.",
    },
    {
        "prompt": "What makes an ablation study useful?",
        "chosen": "A useful ablation changes one factor at a time and keeps the rest of the setup fixed. That helps identify which component actually caused the metric movement.",
        "rejected": "An ablation is useful when you change many things at once and get a better final number.",
    },
    {
        "prompt": "When is BM25 stronger than dense embeddings?",
        "chosen": "BM25 is often stronger for exact tokens such as product names, stack traces, error codes, and rare identifiers. Dense embeddings help more when the query and document use different words with similar meaning.",
        "rejected": "BM25 is always worse because it does not use neural networks.",
    },
    {
        "prompt": "Give a practical 8GB VRAM fine-tuning strategy.",
        "chosen": "Use a small model, short sequences, LoRA, small batches, gradient accumulation, and mixed precision if supported. Measure memory and trainable parameters so the tradeoff is explicit.",
        "rejected": "Use the largest model available and hope Colab automatically handles memory.",
    },
]

dataset = Dataset.from_list(preference_rows).train_test_split(test_size=0.33, seed=SEED)
dataset


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

trainable_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_before = sum(p.numel() for p in model.parameters())
print({"base_trainable_params": trainable_before, "base_total_params": total_before})


In [ ]:
def make_dpo_config():
    kwargs = dict(
        output_dir="dpo-smollm2-mini",
        beta=0.1,
        learning_rate=5e-5,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        max_length=384,
        max_prompt_length=128,
        logging_steps=1,
        save_strategy="no",
        report_to=[],
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )
    strategy_name = "eval_strategy" if "eval_strategy" in inspect.signature(DPOConfig.__init__).parameters else "evaluation_strategy"
    kwargs[strategy_name] = "epoch"
    return DPOConfig(**kwargs)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=make_dpo_config(),
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

dpo_trainer.train()
eval_metrics = dpo_trainer.evaluate()
eval_metrics


In [ ]:
def generate_answer(prompt, max_new_tokens=80):
    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f"User: {prompt}\nAssistant:"
    inputs = tokenizer(text, return_tensors="pt").to(dpo_trainer.model.device)
    with torch.no_grad():
        output = dpo_trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

test_prompts = [
    "Explain DPO versus PPO-style RLHF in two sentences.",
    "Why is LoRA useful on limited VRAM?",
    "Why should RAG evaluation separate retrieval and generation?",
]

pd.DataFrame({"prompt": test_prompts, "model_answer": [generate_answer(p) for p in test_prompts]})


In [ ]:
# Inspect the preference dataset as a product artifact.
pd.DataFrame(preference_rows)


## Talk Track

- DPO uses prompt, chosen answer, and rejected answer triples.
- It directly increases preference for chosen completions over rejected completions.
- Compared with PPO-style RLHF, this avoids a separate reward model and online reinforcement learning loop, which makes it high ROI for a small notebook project.
